# Author Info
---
Name: **Ejaz-ur-Rehman**\
Business Unit Head | Data Analyst | Crystal Tech (Project of MUZHAB Group)\
Faculty Member-Department of Management Sciences | Al Ghazali University\
MS (Finance), PhD Scholar\
Karachi, Pakistan

![Date](https://img.shields.io/badge/Date-24--July--2026-green?logo=google-calendar)
[![Email](https://img.shields.io/badge/Email-ijazfinance%40gmail.com-blue?logo=gmail)](mailto:ijazfinance@gmail.com)
[![LinkedIn](https://img.shields.io/badge/LinkedIn-Ejaz--ur--Rehman-blue?logo=linkedin)](https://www.linkedin.com/in/ejaz-ur-rehman/)
[![GitHub](https://img.shields.io/badge/GitHub-ejazurrehman-black?logo=github)](https://github.com/ejazurrehman)

# Notebook 02: Feature Engineering for Islamic Banking Data


# Learning Objectives

After completing this notebook, students will be able to:

1. Explain feature engineering in finance.
2. Create time-based variables.
3. Generate QoQ and YoY growth.
4. Build lag features.
5. Compute rolling statistics.
6. Apply logarithmic transformations.
7. Detect outliers.
8. Handle missing values.
9. Scale numerical features.
10. Export a modelling-ready dataset.

---

# Research Questions

- Why is feature engineering important?
- Which engineered variables improve predictive models?
- How do temporal features enhance financial forecasting?

---

# Notebook Overview

Feature engineering converts raw financial observations into meaningful analytical variables. This notebook demonstrates a complete feature engineering workflow using Islamic banking data to prepare a high-quality modelling dataset.

---

# Workflow

```text
Raw Dataset
      │
      ▼
Data Understanding
      │
      ▼
Feature Engineering
      │
      ▼
Visualization
      │
      ▼
Business Interpretation
      │
      ▼
Export Modelling Dataset
```

---

## Expected Output

```text
Islamic_Banking_Feature_Engineered.csv
```


## Section 01: Introduction

Before creating new features, analysts must understand the dataset. In financial analytics, every variable has business meaning. Proper data understanding helps prevent modelling errors and ensures that engineered features are both statistically valid and economically meaningful.

In [3]:
import pandas as pd

df = pd.read_csv("Cleaned_Islamic_Banking_Data.csv")

print(df.columns.tolist())

['Country', 'Regional Code', 'Report Type', 'Time Period', 'USD Exchange Rate', 'Indicator Code', 'Indicator Description', 'currency', 'units', 'Values', 'Values in USD Millions']


In [4]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    "Cleaned_Islamic_Banking_Data.csv",
    parse_dates=["Time Period"]
)

print(df.shape)
df.head()

(86789, 11)


C:\Users\DELL\AppData\Local\Temp\ipykernel_25224\1900768544.py:4: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df = pd.read_csv(


,Country,Regional Code,Report Type,Time Period,USD Exchange Rate,Indicator Code,Indicator Description,currency,units,Values,Values in USD Millions
0,Bangladesh,SA,Islamic Banking,2013-10-01,77.4,CP01a,CAR,NaN,NaN,12.06,12.06
1,Bangladesh,SA,Islamic Banking,2013-10-01,77.4,CP01a_010,Total regulatory capital,NC,M,107775.73,1392.45
2,Bangladesh,SA,Islamic Banking,2013-10-01,77.4,CP01a_020,Risk-weighted assets (RWA),NC,M,893983.11,11550.17
3,Bangladesh,SA,Islamic Banking,2013-10-01,77.4,CP02a,Tier 1 capital to RWA,NaN,NaN,9.60,9.60
4,Bangladesh,SA,Islamic Banking,2013-10-01,77.4,CP02a_010,Tier 1 capital,NC,M,85804.37,1108.58


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 86789 entries, 0 to 86788
Data columns (total 11 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   Country                 86789 non-null  object        
 1   Regional Code           86789 non-null  object        
 2   Report Type             86789 non-null  object        
 3   Time Period             86789 non-null  datetime64[ns]
 4   USD Exchange Rate       86789 non-null  float64       
 5   Indicator Code          86789 non-null  object        
 6   Indicator Description   86789 non-null  object        
 7   currency                69510 non-null  object        
 8   units                   71038 non-null  object        
 9   Values                  59458 non-null  float64       
 10  Values in USD Millions  59128 non-null  float64       
dtypes: datetime64[ns](1), float64(3), object(7)
memory usage: 7.3+ MB


## Section 02: Descriptive Statistics

Review the central tendency and dispersion of numerical variables before engineering new features.


In [6]:
df.describe(include="all").T

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
Country,86789,5,Pakistan,18870,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Regional Code,86789,3,EAP,36975,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Report Type,86789,2,Islamic Banking Windows,43522,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Time Period,86789,NaN,NaN,NaN,2019-05-10 22:19:20.193112064,2013-10-01 00:00:00,2016-07-01 00:00:00,2019-04-01 00:00:00,2022-01-01 00:00:00,2025-10-01 00:00:00,NaN
USD Exchange Rate,86789.0,NaN,NaN,NaN,3134.34018,3.21,4.05,83.9,280.19,16421.0,5886.104598
Indicator Code,86789,200,CP01a,453,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Indicator Description,86789,142,Total regulatory capital,2265,NaN,NaN,NaN,NaN,NaN,NaN,NaN
currency,69510,2,NC,69394,NaN,NaN,NaN,NaN,NaN,NaN,NaN
units,71038,3,M,53680,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Values,59458.0,NaN,NaN,NaN,142622.447658,-25843.55,30.64,3043.93,46853.86,9327270.16,501143.289446


Dataset Overview
> The dataset contains 86,789 observations and 11 variables, comprising categorical, datetime, and numerical features. It represents Islamic banking indicators across multiple countries and reporting periods, making it suitable for longitudinal and comparative financial analysis.

## Section 03: Variable Classification

Financial datasets generally contain:

| Variable Type | Example |
|---|---|
| Date | Reporting Date |
| Numerical | Assets, Deposits |
| Categorical | Bank Type |
| Identifier | Bank ID |

Each variable type requires different feature engineering techniques.

In [7]:
numeric = df.select_dtypes(include=np.number).columns.tolist()
categorical = df.select_dtypes(include="object").columns.tolist()
datetime = df.select_dtypes(include="datetime").columns.tolist()

print("Numeric:", numeric)
print("Categorical:", categorical)
print("Datetime:", datetime)

Numeric: ['USD Exchange Rate', 'Values', 'Values in USD Millions']
Categorical: ['Country', 'Regional Code', 'Report Type', 'Indicator Code', 'Indicator Description', 'currency', 'units']
Datetime: ['Time Period']


## Section 04: Missing Value Assessment

Missing observations should be identified before feature engineering.

In [8]:
missing = pd.DataFrame({
    "Missing Values": df.isna().sum(),
    "Percentage (%)": (df.isna().mean()*100).round(2)
})

missing.sort_values("Missing Values", ascending=False)

,Missing Values,Percentage (%)
Values in USD Millions,27661,31.87
Values,27331,31.49
currency,17279,19.91
units,15751,18.15
Report Type,0,0.00
Country,0,0.00
Regional Code,0,0.00
Indicator Description,0,0.00
Indicator Code,0,0.00
USD Exchange Rate,0,0.00


## 4.1 Investigating Missing Values by Indicator

The first step is to determine whether missing values are concentrated in specific financial indicators.

In [9]:
# Missing Values by Indicator

missing_by_indicator = (
    df.groupby("Indicator Description")
      .agg(
          Total_Records=("Values", "size"),
          Missing_Values=("Values", lambda x: x.isna().sum())
      )
)

missing_by_indicator["Missing (%)"] = (
    missing_by_indicator["Missing_Values"] /
    missing_by_indicator["Total_Records"] * 100
).round(2)

missing_by_indicator = missing_by_indicator.sort_values(
    by="Missing (%)",
    ascending=False
)

missing_by_indicator.head(20)

,Total_Records,Missing_Values,Missing (%)
Indicator Description,,,
Value (or percentage) of Sharī`ah-compliant financing by economic activity,453,453,100.00
Value (or percentage) of financing by type of Sharī`ah-compliant contract,453,453,100.00
"of which: fishing, fish farming, aquaculture and related service activities",44,44,100.00
Value (or percentage) of returns by major type of Sharī`ah-compliant contract,453,452,99.78
Value (or percentage) of gross NPF by economic activities,453,448,98.90
Assets held by domestic systemically important Islamic banks,453,428,94.48
Common Equity Tier 1 (CET1) capital to RWA (IFSB),453,416,91.83
CAR (IFSB),453,404,89.18
Tier 1 capital to RWA (IFSB),453,404,89.18


Interpretation: Missing Values by Indicator
> The analysis of missing values at the indicator level reveals that missing observations are highly concentrated within specific financial indicators rather than being randomly distributed across the dataset. Several indicators exhibit extremely high missing percentages, with some reaching 100% missing values. This pattern indicates that these variables were either not reported, were optional under the reporting framework, or were only applicable to selected countries or reporting periods.

For example, the indicators "Value (or percentage) of Sharīah-compliant financing by economic activity"** and **"Value (or percentage) of financing by type of Sharīah-compliant contract" each contain 453 observations, all of which are missing (100%). Similarly, several capital adequacy and liquidity indicators, including CAR (IFSB), Common Equity Tier 1 (CET1) capital to RWA, Tier 1 capital to RWA, and Net Stable Funding Ratio (NSFR), exhibit missing rates exceeding 80%. These findings suggest that such indicators were reported only by a limited number of jurisdictions or during specific reporting periods.
> The presence of missing values concentrated within particular indicators implies that the missingness is structural (systematic) rather than random. Therefore, replacing these missing observations using simple imputation techniques, such as mean, median, or zero substitution, would introduce artificial information and could distort subsequent statistical analyses and machine learning models.
> 
> Based on this evidence, these missing values should be treated as genuine unavailable observations rather than data entry errors. Indicators with exceptionally high missing rates should either be retained as missing values, analyzed separately, or considered for exclusion in modeling tasks if they contribute limited analytical value.

## 4.2 Missing Values by Country

In [10]:
missing_country = (
    df.groupby("Country")["Values"]
      .apply(lambda x: x.isna().mean() * 100)
      .round(2)
      .sort_values(ascending=False)
)

missing_country.to_frame(name="Missing (%)")

,Missing (%)
Country,
Pakistan,43.99
Saudi Arabia,30.46
Bangladesh,29.51
Malaysia,26.49
Indonesia,26.19


Interpretation: Missing Values by Country
> The country-wise analysis reveals that the proportion of missing values differs across the five countries included in the dataset. This variation suggests that the availability of financial information is influenced by differences in regulatory reporting practices, disclosure requirements, or data submission across jurisdictions.
> 
> Among the countries, Pakistan exhibits the highest proportion of missing observations, with 43.99% of the financial values unavailable. This indicates that nearly half of the observations for Pakistan do not contain reported values for the selected financial indicators. The relatively higher missing rate may be attributable to the inclusion of additional indicator categories, differences in reporting coverage, or incomplete disclosure for certain reporting periods.
>
> Saudi Arabia records the second-highest missing percentage (30.46%), followed by Bangladesh (29.51%), Malaysia (26.49%), and Indonesia (26.19%). The comparatively lower missing rates for Malaysia and Indonesia suggest more complete reporting of financial indicators throughout the study period.
>
> Although the extent of missing data varies among countries, all countries exhibit some degree of missingness. This pattern indicates that missing values are not isolated to a single jurisdiction but are a common characteristic of the international Islamic banking dataset. Combined with the indicator-level analysis, these findings suggest that the missing values are largely driven by differences in reporting practices and indicator availability, rather than random data entry errors.
>
> Therefore, missing observations should not be replaced using a single imputation method for the entire dataset. Instead, any treatment should consider both the country and the financial indicator to preserve the accuracy and reliability of subsequent analyses.

## 4.3 Missing Values by Report Type

In [11]:
missing_report = (
    df.groupby("Report Type")["Values"]
      .apply(lambda x: x.isna().mean() * 100)
      .round(2)
)

missing_report.to_frame(name="Missing (%)")

,Missing (%)
Report Type,
Islamic Banking,23.49
Islamic Banking Windows,39.44


Interpretation: Missing Values by Report Type
> The analysis of missing values by report type reveals noticeable differences between the two reporting categories included in the dataset. Islamic Banking Windows exhibits a substantially higher proportion of missing financial values (39.44%) compared with Islamic Banking, which has a missing rate of 23.49%.
>
> The higher percentage of missing values in Islamic Banking Windows suggests that these institutions report fewer financial indicators or have less comprehensive disclosure than fully-fledged Islamic banks. Since Islamic banking windows operate as Islamic business units within conventional banks, they may be subject to different reporting requirements, data availability, or regulatory disclosure standards.
>
> In contrast, Islamic Banking institutions demonstrate a relatively lower level of missing data, indicating more complete reporting across the selected financial indicators. Nevertheless, the presence of missing observations in both report types confirms that incomplete reporting is a common characteristic of the dataset rather than being restricted to a single institutional category.
>
> These findings, together with the previous country-level and indicator-level analyses, indicate that the missing values are primarily associated with differences in reporting practices and institutional characteristics rather than random data loss or data entry errors.
>
> Therefore, missing values should not be treated using a single imputation strategy for the entire dataset. Any missing value treatment should consider the report type, country, and financial indicator to ensure that the integrity of the financial information is preserved.

## 4.4 Missing Pattern Analysis

Now determine whether the two monetary variables are missing together.

In [12]:
missing_pattern = pd.crosstab(
    df["Values"].isna(),
    df["Values in USD Millions"].isna(),
    margins=True
)

missing_pattern

Values in USD Millions,False,True,All
Values,,,
False,59128,330,59458
True,0,27331,27331
All,59128,27661,86789


Interpretation: Missing Pattern Analysis
> The missing pattern analysis examines the relationship between the two primary monetary variables, Values and Values in USD Millions, to determine whether their missing observations occur independently or simultaneously.
>
> The cross-tabulation reveals a strong and highly consistent missingness pattern. Of the 27,331 observations where the Values column is missing, all 27,331 observations also have missing values in Values in USD Millions. There are no observations in which the original financial value is missing while the corresponding USD-converted value is available.
>
> Conversely, among the 59,458 observations where the Values column is available, 59,128 observations also contain the corresponding USD-converted values. Only 330 observations have an original financial value available but lack its USD equivalent. This small number of cases likely reflects situations where currency conversion was not possible because of unavailable exchange rate information or because the reported indicator was not intended to be converted into USD.
>
> Overall, these findings indicate that the missing values in the two monetary variables are highly dependent rather than independent. The USD-denominated values are derived from the original financial values; therefore, when the source value is unavailable, the converted value is naturally unavailable as well. This pattern represents structural missingness, which is expected in financial reporting systems rather than indicating data quality issues.
>
> Consequently, the missing values in these variables should not be imputed using simple statistical methods such as mean, median, or zero replacement. Instead, they should either be retained as missing values or handled using indicator-specific and context-aware techniques during subsequent analysis.

## 4.5 Determine Missingness Type
| Variable               | Missing % | Likely Type | Action      |
| ---------------------- | --------: | ----------- | ----------- |
| Values                 |     31.49 | Investigate | Conditional |
| Values in USD Millions |     31.87 | Structural  | Keep        |
| Currency               |     19.91 | Structural  | Keep        |
| Units                  |     18.15 | Structural  | Keep        |


Overall Conclusion of Section 4: Missing Value Treatment
> The analyses by indicator, country, report type, and missing pattern consistently indicate that the missing values in this dataset are predominantly structural (Missing Not at Random - MNAR). They arise from differences in reporting practices, regulatory requirements, and the availability of specific financial indicators across countries and institutional types, rather than from data entry errors or random omissions.
>
> Therefore, a blanket imputation strategy is not appropriate. Instead, missing values should be addressed selectively based on the financial meaning of each indicator and the analytical objective. This approach preserves the integrity and reliability of the dataset for subsequent feature engineering, statistical analysis, and machine learning applications.

## Section 5: Handling Missing Values

Recommended Treatment Strategy
| Variable               | Missing % | Recommended Action | Reason                                    |
| ---------------------- | --------: | ------------------ | ----------------------------------------- |
| Country                |        0% | Keep               | Complete                                  |
| Regional Code          |        0% | Keep               | Complete                                  |
| Report Type            |        0% | Keep               | Complete                                  |
| Time Period            |        0% | Keep               | Complete                                  |
| Indicator Code         |        0% | Keep               | Complete                                  |
| Indicator Description  |        0% | Keep               | Complete                                  |
| Currency               |    19.91% | Keep `NaN`         | Not all indicators have a currency        |
| Units                  |    18.15% | Keep `NaN`         | Ratios and percentages often have no unit |
| Values                 |    31.49% | Treat selectively  | Depends on the indicator                  |
| Values in USD Millions |    31.87% | Treat selectively  | Derived from `Values`                     |

Notice that `currency and units` should not be filled.

## Step 5.1: Remove Completely Empty Indicators

Our analysis showed indicators such as:

Value (or percentage) of Sharīʿah-compliant financing by economic activity
Value (or percentage) of financing by type of Sharīʿah-compliant contract

with 100% missing values.

These variables contain no usable information.

In [13]:
# Identify indicators with 100% missing Values
empty_indicators = (
    missing_by_indicator[
        missing_by_indicator["Missing (%)"] == 100
    ].index
)

# Remove them
df = df[~df["Indicator Description"].isin(empty_indicators)]

## Step 5.2: Handle Time-Series Gaps

For indicators with only a few missing observations, interpolate within each Country × Indicator.

In [14]:
df = df.sort_values(
    ["Country", "Indicator Description", "Time Period"]
)

df["Values"] = (
    df.groupby(["Country", "Indicator Description"])["Values"]
      .transform(lambda x: x.interpolate())
)

## Step 5.3: Recalculate USD Values

If Values in `USD Millions` is simply derived from `Values` and the e`xchange rate`, recompute it where possible rather than imputing it independently.

In [15]:
mask = (
    df["Values"].notna() &
    df["USD Exchange Rate"].notna() &
    df["Values in USD Millions"].isna()
)

df.loc[mask, "Values in USD Millions"] = (
    df.loc[mask, "Values"] /
    df.loc[mask, "USD Exchange Rate"]
)

## Step 5.4: Retain Remaining Missing Values

If values are still missing after the previous steps, leave them as NaN. They reflect unavailable information, and many statistical methods and machine learning algorithms can handle missing values appropriately.

## Step 5.5: Verify the Results

In [16]:
missing_after = pd.DataFrame({
    "Missing Values": df.isna().sum(),
    "Percentage (%)": (df.isna().mean() * 100).round(2)
})

missing_after.sort_values("Percentage (%)", ascending=False)

,Missing Values,Percentage (%)
currency,17191,20.03
units,15663,18.25
Values in USD Millions,11993,13.97
Values,11993,13.97
Report Type,0,0.00
Country,0,0.00
Regional Code,0,0.00
Indicator Description,0,0.00
Indicator Code,0,0.00
USD Exchange Rate,0,0.00


Interpretation of Missing Values After Treatment
> The missing value treatment substantially improved the completeness of the primary financial variables. The proportion of missing observations in both Values and Values in USD Millions decreased from approximately 31% to 14%, indicating that a significant number of genuine missing observations were successfully recovered through indicator-specific treatment.
>
> The remaining missing values are primarily associated with currency and units, which continue to exhibit missing rates of approximately 20% and 18%, respectively. Based on the earlier indicator-level analysis, these missing values are considered structural because many financial ratios, percentages, and non-monetary indicators do not require a currency or measurement unit. Therefore, no further treatment is recommended for these variables.
>
> Similarly, the remaining 13.97% missing observations in both Values and Values in USD Millions are concentrated within indicators that were either not reported or not applicable in specific countries or reporting periods. These observations represent genuine data unavailability rather than data quality issues and should therefore be retained as missing values.
>
> Overall, the missing value treatment has significantly enhanced the dataset while preserving the integrity of the original financial information. The cleaned dataset is now suitable for feature engineering and subsequent statistical and machine learning analyses.

## Final Recommendation

We would stop here.

Do not attempt to reduce the remaining 13.97% missing values to zero because:

- They are structural (MNAR).
- Filling them with mean, median, or zero would introduce bias.
- Interpolation is not appropriate for indicators that were never reported.

This is a realistic outcome for regulatory financial datasets.

## Section 6: Feature Engineering

## 6.1 Date-Based Feature Engineering
Objective
> Extract temporal components from the Time Period variable to facilitate trend analysis, seasonal analysis, quarterly reporting, and time-series modeling.

In [17]:
df["Year"] = df["Time Period"].dt.year
df["Quarter"] = df["Time Period"].dt.quarter
df["Month"] = df["Time Period"].dt.month
df["Month_Name"] = df["Time Period"].dt.month_name()
df["Year_Quarter"] = (
    df["Year"].astype(str)
    + "-Q"
    + df["Quarter"].astype(str)
)

## Verification

In [18]:
df[
    [
        "Time Period",
        "Year",
        "Quarter",
        "Month",
        "Month_Name",
        "Year_Quarter"
    ]
].head()

,Time Period,Year,Quarter,Month,Month_Name,Year_Quarter
84,2013-10-01,2013,4,10,October,2013-Q4
111,2013-10-01,2013,4,10,October,2013-Q4
43351,2013-10-01,2013,4,10,October,2013-Q4
43378,2013-10-01,2013,4,10,October,2013-Q4
279,2014-01-01,2014,1,1,January,2014-Q1


Interpretation
> The newly created temporal features provide multiple levels of time aggregation that support annual, quarterly, and monthly analyses. These variables will be useful for identifying long-term trends, seasonal patterns, and period-wise variations in Islamic banking performance. The Year_Quarter feature also provides a convenient time index for quarterly financial reporting and visualization.

## 6.2 Indicator Categorization

Instead of having nearly 200 separate indicators, classify them into financial domains.

In [19]:
def classify_indicator(text):

    text = str(text).lower()

    if any(word in text for word in
           ["asset", "deposit", "financing", "investment"]):
        return "Balance Sheet"

    elif any(word in text for word in
             ["roa", "roe", "profit", "income", "return"]):
        return "Profitability"

    elif any(word in text for word in
             ["capital", "car", "tier", "cet1"]):
        return "Capital Adequacy"

    elif any(word in text for word in
             ["liquidity", "cash", "lcr", "nsfr",
              "stable funding"]):
        return "Liquidity"

    elif any(word in text for word in
             ["npf", "non-performing", "impaired"]):
        return "Asset Quality"

    elif any(word in text for word in
             ["efficiency", "cost", "expense"]):
        return "Efficiency"

    else:
        return "Other"


df["Indicator Category"] = (
    df["Indicator Description"]
    .apply(classify_indicator)
)

## Verification

In [20]:
df["Indicator Category"].value_counts()

Indicator Category
Other               48460
Balance Sheet       17447
Capital Adequacy     9966
Profitability        4983
Liquidity            2265
Asset Quality        2265
Efficiency            453
Name: count, dtype: int64

Interpretation
> Grouping detailed financial indicators into broader analytical categories reduces dimensionality and improves interpretability. The engineered variable enables comparative analyses across major areas of Islamic banking performance, including balance sheet structure, profitability, liquidity, capital adequacy, asset quality, and operational efficiency.

## 6.3 Data Type Optimization

In [21]:
category_columns = [

    "Country",
    "Regional Code",
    "Report Type",
    "currency",
    "units",
    "Indicator Category",
    "Month_Name"

]

for col in category_columns:
    df[col] = df[col].astype("category")

## Memory Comparison

In [22]:
print(df.info(memory_usage="deep"))

<class 'pandas.core.frame.DataFrame'>
Index: 85839 entries, 84 to 86781
Data columns (total 17 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   Country                 85839 non-null  category      
 1   Regional Code           85839 non-null  category      
 2   Report Type             85839 non-null  category      
 3   Time Period             85839 non-null  datetime64[ns]
 4   USD Exchange Rate       85839 non-null  float64       
 5   Indicator Code          85839 non-null  object        
 6   Indicator Description   85839 non-null  object        
 7   currency                68648 non-null  category      
 8   units                   70176 non-null  category      
 9   Values                  73846 non-null  float64       
 10  Values in USD Millions  73846 non-null  float64       
 11  Year                    85839 non-null  int32         
 12  Quarter                 85839 non-null  int32     

Interpretation
> Converting low-`cardinality` text variables to the category data type reduces memory consumption and improves computational efficiency during grouping, aggregation, and machine learning tasks. This optimization is particularly beneficial for large financial datasets containing repeated categorical values.
>
> Cardinality: 

## 6.4 Time-Series Features

Rather than creating arbitrary machine learning features, create features that directly support statistical analysis.

Observation Frequency

In [23]:
# Number of observations per indicator

obs_frequency = (
    df.groupby("Indicator Description")
      .size()
      .reset_index(name="Frequency")
)

df = df.merge(
    obs_frequency,
    on="Indicator Description",
    how="left"
)

## Country Indicator Count

In [24]:
country_indicator_count = (
    df.groupby("Country")["Indicator Description"]
      .transform("nunique")
)

df["Country_Indicator_Count"] = (
    country_indicator_count
)

C:\Users\DELL\AppData\Local\Temp\ipykernel_25224\949550513.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("Country")["Indicator Description"]


## Verification

In [25]:
df[
    [
        "Country",
        "Indicator Description",
        "Frequency",
        "Country_Indicator_Count"
    ]
].head()

,Country,Indicator Description,Frequency,Country_Indicator_Count
0,Bangladesh,"(a) agriculture, forestry, hunting and fishing",906,136
1,Bangladesh,"(a) agriculture, forestry, hunting and fishing",906,136
2,Bangladesh,"(a) agriculture, forestry, hunting and fishing",906,136
3,Bangladesh,"(a) agriculture, forestry, hunting and fishing",906,136
4,Bangladesh,"(a) agriculture, forestry, hunting and fishing",906,136


Interpretation
> These analytical features quantify the reporting frequency of each financial indicator and the breadth of indicator coverage within each country. They provide valuable `metadata` for subsequent statistical analyses, allowing comparisons of reporting intensity, indicator availability, and cross-country reporting completeness.
>
> Metadata: 

## Finally, save the engineered dataset for the next working.

In [26]:
df.to_csv(
    "Islamic_Banking_Feature_Engineered.csv",
    index=False
)

print("Feature engineered dataset saved successfully.")

Feature engineered dataset saved successfully.


## **NOTE**
> This notebook creates features with a clear analytical purpose. Each engineered variable will be used later for statistical testing, trend analysis, visualization, or machine learning, avoiding unnecessary or redundant feature creation.